# RAG Experiments | DiagnosAI Medical Assistant

## End-to-end RAG pipeline built on 170 WHO disease fact sheets.

**Overview**

Enables doctors and patients to ask deep medical questions beyond the structured ML prediction model grounded in verified WHO data.

Documents used in the Knowledge base are scraped from: https://www.who.int/news-room/fact-sheets/

Stack Used:
* LangChain
* FAISS
* Google Gemini
* Python

The following approach will be taken:

1. Document Loading
2. Chunking
3. Embeddings
4. Vector Store (FAISS)
5. Retrieval
6. Generation
7. Experimentation

## Preparing the tools

In [2]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

## 1. Load documents

In [4]:
loader = DirectoryLoader(
    path="data/processed/knowledge_base",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"})

documents = loader.load()
print(f"Loading {len(documents)} documents ...")

Loading 170 documents ...


## 2. Chunking

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)
print(f"Total chunks: {len(chunks)}")
print(f"\nSample chunk:\n{chunks[0].page_content}")

Total chunks: 2097

Sample chunk:
Key facts
Six out of 10 unintended pregnancies end in induced abortion.
Abortion is a common health intervention. It is very safe when carried out using a method recommended by WHO, appropriate to the pregnancy duration and by someone with the necessary skills.
However, around 45% of abortions are unsafe.
Unsafe abortion is an important preventable cause of maternal deaths and morbidities. It can lead to physical and mental health complications and social and financial burdens for women, communities and health systems.
Lack of access to safe, timely, affordable and respectful abortion care is a critical public health and human rights issue.
Overview
Around 73 million induced abortions take place worldwide each year. Six out of 10 (61%) of all unintended pregnancies, and 3 out of 10 (29%) of all pregnancies, end in induced abortion(1).


## 3. Embeddings


In [7]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 4. Vector Store (FAISS)

In [8]:
vectorstore = FAISS.from_documents(chunks, embeddings)

vectorstore.save_local("artifacts/faiss_index")

## 5. Retrieval

In [10]:
vectorstore = FAISS.load_local("artifacts/faiss_index", embeddings, allow_dangerous_deserialization=True)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k" : 4}
)

docs = retriever.invoke("Talk about Cancer?")
for doc in docs:
    print(doc.page_content)
    print(doc.metadata)


Key facts
Cancer is a leading cause of death worldwide, accounting for nearly 10 million deaths in 2024, or nearly one in six deaths.
The most common cancers are lung, breast, colon and rectum and prostate cancers.
Nearly a quarter of deaths from cancer are due
to tobacco use, alcohol consumption, high body mass index, low fruit and
vegetable intake, and lack of physical activity. In addition, air pollution is an important risk factor for lung cancer.
Cancer-causing infections, such as human
papillomavirus (HPV) and hepatitis, are responsible for approximately
30% of cancer cases in low- and lower-middle-income countries.
Many cancers can be cured if detected early and treated effectively.
Overview
Cancer is a generic term for a large group of diseases that can affect any part of the body. Other terms used are malignant tumours and neoplasms. One defining
{'source': 'data/processed/knowledge_base/Cancer.txt'}
liver (732 000 deaths)
breast (694 000 deaths)
stomach (642 000 deaths).
The 

## 6. Generation

In [16]:
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.3
)

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are DiagnosAI, a trustworthy medical assistant. Your tone is cool, welcoming, and professional—never cold, robotic, or overly sympathetic. Your tone should be of a human being trying to help out by giving information. DO NOT SOUND ROBOTIC AND COLD.

Your highest responsibility is to answer questions accurately and safely using ONLY the information contained in the provided context.

RULES:

1. USE ONLY THE PROVIDED CONTEXT
- Every medical fact, symptom, treatment, recommendation, or statistic must come directly from the context block below.
- Do not use outside medical knowledge, and do not guess or extrapolate.

2. ANSWER STYLE & TONE
- Speak directly to the patient using clear, conversational, and patient-friendly language.
- A casual, welcoming greeting like "Hi" or "Hello" is fine to use at the beginning.
- Summarize and explain information in your own words instead of copying the context verbatim.
- Keep the answer well-organized using short paragraphs or bullet points when appropriate.
- Maintain a balanced, calm clinical presence. Do not apologize or use overly emotional or sympathetic language.

At the end of your response, provide exactly 2 relevant follow-up questions the user might want to ask next, formatted as:
Suggestions:
1. [Follow-up question 1]
2. [Follow-up question 2]

3. WHEN INFORMATION IS MISSING
If the context does not contain the answer to the question, do not guess or try to comfort them. Reply exactly with this phrase and nothing else:
"I don't have information about that in my knowledge base."

----------------
[START OF CONTEXT]
{context}
[END OF CONTEXT]
----------------

User Question: {question}

Helpful Answer:
"""
)

chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt" : prompt_template},
    return_source_documents=True
)

result = chain.invoke({"query": "Talk about Typhoid in details"})
print(result["result"])

Hi there. I can certainly walk you through the details regarding typhoid fever based on the information I have.

**What is it?**
Typhoid fever is a life-threatening infection caused by the *Salmonella* Typhi bacteria. It is typically contracted by consuming contaminated food or water. Once the bacteria are ingested, they multiply and move into the bloodstream. Because the bacteria live only in humans, those infected carry the germs in their bloodstream and intestinal tract.

**Symptoms to look for**
The primary symptoms include:
*   A prolonged, high fever.
*   Fatigue and headaches.
*   Nausea and abdominal pain.
*   Constipation or diarrhea.
*   In some cases, a rash may appear.

It is important to note that severe cases can lead to serious complications or even death. If you suspect you have it, the condition is confirmed through a blood test.

**Treatment and recovery**
While typhoid is treated with antibiotics, it is becoming more complicated to manage because the bacteria are inc

## 6. Experimentation

In [21]:
def answer_question(question: str) -> str:
    result = chain.invoke({"query": question})
    return result["result"]

print(answer_question("What are the symptoms of Malaria?"))
print(answer_question("How is Typhoid treated?"))
print(answer_question("What causes Diabetes?"))
print(answer_question("Outline the symtoms of Tuberculosis?"))

Hi! If you're looking into the symptoms of malaria, here is what you should know:

The early signs usually show up about 10 to 15 days after being bitten by an infected mosquito. The most common early symptoms include:
*   Fever
*   Headache
*   Chills

Because these early signs can feel like many other illnesses, it can be difficult to recognize them as malaria, which is why getting tested early is important.

In some cases, the illness can become severe. You should seek emergency care immediately if you experience any of these severe symptoms:
*   Extreme fatigue or tiredness
*   Difficulty breathing
*   Jaundice (yellowing of the skin or eyes)
*   Impaired consciousness
*   Multiple convulsions
*   Dark or bloody urine
*   Abnormal bleeding

Getting treatment early for mild cases is the best way to prevent the infection from progressing to these severe stages.
Hi! Typhoid fever is treated using antibiotics. 

It is very important that you take the full course of antibiotics exactly 